# Speaker id for 5.1

In [3]:
# ============================================================
# SPEAKER-INDEPENDENT DATASET SPLIT
# ============================================================
#
# LOGIC
#
# 1. ViToSA
#    -> GIỮ NGUYÊN train / validation / test
#
# 2. FULL SOURCES
#    -> Common Voice
#    -> Retrieval
#
#    -> DÙNG TOÀN BỘ
#    -> Speaker là atomic group
#    -> Speaker đã được assign split nào
#       thì source khác của cùng speaker cũng theo split đó
#
# 3. FOSD
#    -> CHỈ dùng để compensation non-toxic
#    -> Không tự động được kéo vào train/validation
#       chỉ vì trùng speaker với FULL SOURCE
#
#    -> Nếu train:
#          non-toxic < toxic
#       thì lấy FOSD speaker có lợi thế non-toxic
#
#    -> Nếu validation:
#          non-toxic < toxic
#       thì tiếp tục lấy FOSD speaker
#
#    -> FOSD còn dư:
#          external_non_toxic
#
# 4. IMPORTANT
#
#    Speaker là atomic group.
#
#    Không được có:
#
#        speaker A -> train
#        speaker A -> validation
#
#    Final requirement:
#
#        train_speakers ∩ validation_speakers = ∅
#
# ============================================================


# ============================================================
# 0. IMPORT
# ============================================================

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import soundfile as sf

from tqdm.auto import tqdm

from speechbrain.inference.speaker import EncoderClassifier

from sklearn.cluster import AgglomerativeClustering


# ============================================================
# 1. CONFIG
# ============================================================

# ------------------------------------------------------------
# INPUT CSV
# ------------------------------------------------------------

INPUT_CSV = Path(
    r"A:\A _ Working\Researching\B - AIoT Lab VN\VITOSA SpeechRun\vitosa_datasets\final_vietnamese_toxic_utterance_dataset_v5.1.csv"
)


# ------------------------------------------------------------
# AUDIO FOLDER
# ------------------------------------------------------------

WAV_FOLDER = Path(
    r"A:\A _ Working\Researching\B - AIoT Lab VN\VITOSA SpeechRun\vitosa_datasets\wav_segments_v5.1"
)


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_CSV = INPUT_CSV.parent / (
    "final_vietnamese_toxic_utterance_dataset_v5.1_speaker_split.csv"
)


# ------------------------------------------------------------
# SKIPPED AUDIO REPORT
# ------------------------------------------------------------

SKIPPED_OUTPUT_CSV = INPUT_CSV.parent / (
    "speaker_split_skipped_audio5.1.csv"
)


# ------------------------------------------------------------
# RANDOM SEED
# ------------------------------------------------------------

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)


# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

device_str = (
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)


# ------------------------------------------------------------
# ECAPA CONFIG
# ------------------------------------------------------------

ECAPA_SOURCE = "speechbrain/spkrec-ecapa-voxceleb"


# ------------------------------------------------------------
# SPEAKER CLUSTERING
# ------------------------------------------------------------

SPEAKER_DISTANCE_THRESHOLD = 0.35


# ------------------------------------------------------------
# REQUIRED SAMPLE RATE
# ------------------------------------------------------------

TARGET_SAMPLE_RATE = 16000


# ------------------------------------------------------------
# MINIMUM AUDIO LENGTH
# ------------------------------------------------------------

MIN_AUDIO_SECONDS = 1.0

MIN_AUDIO_SAMPLES = int(
    TARGET_SAMPLE_RATE * MIN_AUDIO_SECONDS
)


# ============================================================
# 2. PRINT DEVICE
# ============================================================

print("=" * 80)
print("DEVICE")
print("=" * 80)

print(f"Device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():

    print(
        f"GPU: "
        f"{torch.cuda.get_device_name(0)}"
    )


# ============================================================
# 3. LOAD DATASET
# ============================================================

print("\n" + "=" * 80)
print("LOADING DATASET")
print("=" * 80)

if not INPUT_CSV.exists():

    raise FileNotFoundError(
        f"INPUT_CSV không tồn tại:\n{INPUT_CSV}"
    )


df = pd.read_csv(INPUT_CSV)

# ------------------------------------------------------------
# Reset pandas index
# ------------------------------------------------------------

df = df.reset_index(drop=True)


# ------------------------------------------------------------
# Create stable row ID
# ------------------------------------------------------------

df["__row_id"] = np.arange(
    len(df),
    dtype=np.int64
)


# ------------------------------------------------------------
# Required columns
# ------------------------------------------------------------

required_columns = [
    "audio_path",
    "audio_source",
    "toxicity",
    "split",
]


missing_columns = [
    col
    for col in required_columns
    if col not in df.columns
]


if missing_columns:

    raise ValueError(
        "Dataset thiếu các column bắt buộc:\n"
        + "\n".join(
            f"- {col}"
            for col in missing_columns
        )
    )


print(
    f"Total samples: {len(df):,}"
)


# ============================================================
# 4. DATASET SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("AUDIO SOURCE")
print("=" * 80)

print(
    df["audio_source"]
    .value_counts(dropna=False)
)


print("\n" + "=" * 80)
print("ORIGINAL SPLIT")
print("=" * 80)

print(
    df["split"]
    .value_counts(dropna=False)
)


print("\n" + "=" * 80)
print("SOURCE × TOXICITY")
print("=" * 80)

print(
    pd.crosstab(
        df["audio_source"],
        df["toxicity"],
        margins=True,
        margins_name="Tổng_Cộng"
    )
)


# ============================================================
# 5. CREATE pseudo_speaker_id
# ============================================================

df["pseudo_speaker_id"] = pd.NA


# ============================================================
# 6. SOURCE CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# BASE SOURCE
# ------------------------------------------------------------

BASE_SOURCE = "vitosa"


# ------------------------------------------------------------
# FULL SOURCES
# ------------------------------------------------------------

FULL_SOURCE_ORDER = [
    "common_voice",
    "retrieval",
]


# ------------------------------------------------------------
# COMPENSATION SOURCE
# ------------------------------------------------------------

FOSD_SOURCE = "fosd"


# ------------------------------------------------------------
# Available sources
# ------------------------------------------------------------

available_sources = (
    df["audio_source"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)


# ------------------------------------------------------------
# FULL SOURCES that exist
# ------------------------------------------------------------

FULL_SOURCES = [
    source
    for source in FULL_SOURCE_ORDER
    if source in available_sources
]


# ------------------------------------------------------------
# Unknown sources
#
# Any source other than:
#
#     ViToSA
#     FOSD
#
# is considered FULL SOURCE
# ------------------------------------------------------------

UNKNOWN_FULL_SOURCES = [
    source
    for source in available_sources
    if (
        source != BASE_SOURCE
        and source != FOSD_SOURCE
        and source not in FULL_SOURCES
    )
]


FULL_SOURCES.extend(
    UNKNOWN_FULL_SOURCES
)


# ------------------------------------------------------------
# Target sources for ECAPA
# ------------------------------------------------------------

TARGET_SOURCES = (
    FULL_SOURCES
    + (
        [FOSD_SOURCE]
        if FOSD_SOURCE in available_sources
        else []
    )
)


print("\n" + "=" * 80)
print("SOURCE CONFIGURATION")
print("=" * 80)

print(
    f"Base source: "
    f"{BASE_SOURCE}"
)

print("\nFULL SOURCES:")

for i, source in enumerate(
    FULL_SOURCES,
    start=1
):

    print(
        f"  {i:02d}. {source}"
    )


print("\nCOMPENSATION SOURCE:")

print(
    f"  {FOSD_SOURCE}"
)


# ============================================================
# 7. TARGET DATA FOR SPEAKER CLUSTERING
# ============================================================

target_df = df[
    df["audio_source"].isin(
        TARGET_SOURCES
    )
].copy()


print("\n" + "=" * 80)
print("TARGET DATA FOR SPEAKER CLUSTERING")
print("=" * 80)

print(
    f"Target samples: "
    f"{len(target_df):,}"
)


print(
    target_df["audio_source"]
    .value_counts()
)


# ============================================================
# 8. BUILD WAV INDEX
# ============================================================

print("\n" + "=" * 80)
print("BUILDING WAV INDEX")
print("=" * 80)


if not WAV_FOLDER.exists():

    raise FileNotFoundError(
        f"WAV_FOLDER không tồn tại:\n{WAV_FOLDER}"
    )


print(
    f"Scanning:\n{WAV_FOLDER}"
)


wav_files = list(
    WAV_FOLDER.rglob("*.wav")
)


wav_index = {}


for wav_path in wav_files:

    filename = wav_path.name.lower()

    if filename not in wav_index:

        wav_index[filename] = wav_path


print(
    f"Found WAV files: "
    f"{len(wav_files):,}"
)


# ============================================================
# 9. LOAD ECAPA-TDNN
# ============================================================

print("\n" + "=" * 80)
print("LOADING ECAPA-TDNN")
print("=" * 80)


classifier = EncoderClassifier.from_hparams(
    source=ECAPA_SOURCE,
    run_opts={
        "device": device_str
    }
)


print("ECAPA-TDNN loaded successfully.")


# ============================================================
# 10. EXTRACT SPEAKER EMBEDDINGS
# ============================================================

embeddings = []

metadata = []

skipped_audio = []


print("\n" + "=" * 80)
print("EXTRACTING SPEAKER EMBEDDINGS")
print("=" * 80)


for idx, row in tqdm(
    target_df.iterrows(),
    total=len(target_df),
    desc="ECAPA"
):

    # --------------------------------------------------------
    # Stable pandas index
    # --------------------------------------------------------

    df_index = int(idx)


    # --------------------------------------------------------
    # Audio filename
    # --------------------------------------------------------

    audio_filename = Path(
        str(row["audio_path"])
    ).name


    wav_path = wav_index.get(
        audio_filename.lower()
    )


    # --------------------------------------------------------
    # AUDIO NOT FOUND
    # --------------------------------------------------------

    if wav_path is None:

        skipped_audio.append(
            {
                "df_index": df_index,
                "audio_path": audio_filename,
                "audio_source": row[
                    "audio_source"
                ],
                "reason": "audio_not_found",
            }
        )

        continue


    try:

        # ----------------------------------------------------
        # LOAD WAV
        # ----------------------------------------------------

        waveform, sample_rate = sf.read(
            str(wav_path),
            dtype="float32"
        )


        # ----------------------------------------------------
        # MONO
        # ----------------------------------------------------

        if waveform.ndim > 1:

            waveform = waveform.mean(
                axis=1
            )


        # ----------------------------------------------------
        # SAMPLE RATE
        # ----------------------------------------------------

        if sample_rate != TARGET_SAMPLE_RATE:

            skipped_audio.append(
                {
                    "df_index": df_index,
                    "audio_path": audio_filename,
                    "audio_source": row[
                        "audio_source"
                    ],
                    "reason": (
                        f"invalid_sr_{sample_rate}"
                    ),
                }
            )

            continue


        # ----------------------------------------------------
        # TOO SHORT
        # ----------------------------------------------------

        if len(waveform) < MIN_AUDIO_SAMPLES:

            skipped_audio.append(
                {
                    "df_index": df_index,
                    "audio_path": audio_filename,
                    "audio_source": row[
                        "audio_source"
                    ],
                    "reason": "audio_too_short",
                }
            )

            continue


        # ----------------------------------------------------
        # TORCH
        # ----------------------------------------------------

        wav_tensor = torch.tensor(
            waveform,
            dtype=torch.float32
        )


        # ----------------------------------------------------
        # Add batch dimension
        # ----------------------------------------------------

        wav_input = (
            wav_tensor
            .unsqueeze(0)
            .to(device)
        )


        # ----------------------------------------------------
        # ECAPA
        # ----------------------------------------------------

        with torch.no_grad():

            embedding = (
                classifier
                .encode_batch(
                    wav_input
                )
                .squeeze()
                .detach()
                .cpu()
                .numpy()
            )


        # ----------------------------------------------------
        # Validate embedding
        # ----------------------------------------------------

        if embedding.ndim != 1:

            embedding = embedding.reshape(-1)


        if not np.isfinite(
            embedding
        ).all():

            skipped_audio.append(
                {
                    "df_index": df_index,
                    "audio_path": audio_filename,
                    "audio_source": row[
                        "audio_source"
                    ],
                    "reason": (
                        "invalid_embedding"
                    ),
                }
            )

            continue


        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------

        embeddings.append(
            embedding
        )


        metadata.append(
            {
                "df_index": df_index,
                "audio_path": audio_filename,
                "audio_source": row[
                    "audio_source"
                ],
                "toxicity": int(
                    row["toxicity"]
                ),
            }
        )


    except Exception as e:

        skipped_audio.append(
            {
                "df_index": df_index,
                "audio_path": audio_filename,
                "audio_source": row[
                    "audio_source"
                ],
                "reason": (
                    f"error_{type(e).__name__}: "
                    f"{str(e)}"
                ),
            }
        )


# ============================================================
# 11. EMBEDDING EXTRACTION SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("EMBEDDING EXTRACTION COMPLETED")
print("=" * 80)


print(
    f"Valid embeddings : "
    f"{len(embeddings):,}"
)


print(
    f"Skipped audio    : "
    f"{len(skipped_audio):,}"
)


# ------------------------------------------------------------
# Save skipped report
# ------------------------------------------------------------

if skipped_audio:

    skipped_df = pd.DataFrame(
        skipped_audio
    )

    skipped_df.to_csv(
        SKIPPED_OUTPUT_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"\nSkipped report saved:\n"
        f"{SKIPPED_OUTPUT_CSV}"
    )


if len(embeddings) == 0:

    raise RuntimeError(
        "Không có embedding nào được tạo."
    )


# ============================================================
# 12. CREATE EMBEDDING DATAFRAME
# ============================================================

X = np.asarray(
    embeddings,
    dtype=np.float32
)


embedding_df = pd.DataFrame(
    metadata
)


print("\nEmbedding shape:")

print(
    X.shape
)


# ============================================================
# 13. CLUSTER SPEAKERS
# ============================================================

print("\n" + "=" * 80)
print("CLUSTERING SPEAKERS")
print("=" * 80)


cluster_model = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=(
        SPEAKER_DISTANCE_THRESHOLD
    ),
    metric="cosine",
    linkage="average",
)


cluster_labels = (
    cluster_model
    .fit_predict(X)
)


embedding_df["cluster_id"] = (
    cluster_labels
)


print(
    f"Detected pseudo speakers: "
    f"{embedding_df['cluster_id'].nunique():,}"
)


# ============================================================
# 14. GLOBAL pseudo_speaker_id
# ============================================================

embedding_df[
    "pseudo_speaker_id"
] = (
    embedding_df[
        "cluster_id"
    ]
    .apply(
        lambda x:
        f"SPEAKER_{int(x):05d}"
    )
)


# ============================================================
# 15. WRITE pseudo_speaker_id BACK
# ============================================================

for _, row in embedding_df.iterrows():

    df.loc[
        int(row["df_index"]),
        "pseudo_speaker_id"
    ] = row[
        "pseudo_speaker_id"
    ]


# ============================================================
# 16. SPEAKER SUMMARY
# ============================================================

speaker_summary = (
    embedding_df
    .groupby(
        "pseudo_speaker_id"
    )
    .agg(
        samples=(
            "df_index",
            "count"
        ),
        sources=(
            "audio_source",
            lambda x:
            ",".join(
                sorted(
                    set(x)
                )
            )
        ),
        toxic=(
            "toxicity",
            "sum"
        ),
        non_toxic=(
            "toxicity",
            lambda x:
            int(
                (x == 0).sum()
            )
        ),
    )
    .reset_index()
)


speaker_summary["delta"] = (
    speaker_summary["non_toxic"]
    - speaker_summary["toxic"]
)


print("\n" + "=" * 80)
print("SPEAKER SUMMARY")
print("=" * 80)


print(
    speaker_summary.head(20)
)


# ============================================================
# 17. KEEP ORIGINAL VITOSA SPLITS
# ============================================================

vitosa_df = df[
    df["audio_source"]
    == BASE_SOURCE
].copy()


print("\n" + "=" * 80)
print("ORIGINAL VITOSA SPLIT")
print("=" * 80)


print(
    pd.crosstab(
        vitosa_df["split"],
        vitosa_df["toxicity"],
        margins=True,
        margins_name="Tổng_Cộng"
    )
)


# ============================================================
# 18. HELPER FUNCTIONS
# ============================================================

def get_split_counts(
    split_name
):

    subset = df[
        df["split"]
        == split_name
    ]

    toxic = int(
        (
            subset["toxicity"]
            == 1
        ).sum()
    )

    non_toxic = int(
        (
            subset["toxicity"]
            == 0
        ).sum()
    )

    total = (
        toxic
        + non_toxic
    )

    return (
        toxic,
        non_toxic,
        total
    )


def get_deficit(
    split_name
):

    toxic, non_toxic, _ = (
        get_split_counts(
            split_name
        )
    )

    return max(
        0,
        toxic - non_toxic
    )


def print_balance(
    title
):

    train_toxic, train_non_toxic, train_total = (
        get_split_counts("train")
    )

    val_toxic, val_non_toxic, val_total = (
        get_split_counts("validation")
    )


    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


    print(
        f"TRAIN       "
        f"total={train_total:,} | "
        f"toxic={train_toxic:,} | "
        f"non-toxic={train_non_toxic:,} | "
        f"non-toxic/toxic="
        f"{train_non_toxic / max(train_toxic, 1):.3f}"
    )


    print(
        f"VALIDATION  "
        f"total={val_total:,} | "
        f"toxic={val_toxic:,} | "
        f"non-toxic={val_non_toxic:,} | "
        f"non-toxic/toxic="
        f"{val_non_toxic / max(val_toxic, 1):.3f}"
    )


def assign_source_speaker(
    source,
    speaker_id,
    split_name
):
    """
    Chỉ assign ROWS thuộc source hiện tại
    của speaker hiện tại.

    Không kéo source khác theo.

    Đây là điểm quan trọng để FOSD
    không bị kéo vào train/validation
    chỉ vì speaker trùng với FULL SOURCE.
    """

    mask = (
        (
            embedding_df[
                "audio_source"
            ]
            == source
        )
        &
        (
            embedding_df[
                "pseudo_speaker_id"
            ]
            == speaker_id
        )
    )


    indices = (
        embedding_df.loc[
            mask,
            "df_index"
        ]
        .astype(int)
        .tolist()
    )


    if not indices:

        return 0


    df.loc[
        indices,
        "split"
    ] = split_name


    return len(indices)


# ============================================================
# 19. RESET NON-VITOSA
# ============================================================

print("\n" + "=" * 80)
print("RESET NON-VITOSA SOURCES")
print("=" * 80)


non_vitosa_mask = (
    df["audio_source"]
    != BASE_SOURCE
)


df.loc[
    non_vitosa_mask,
    "split"
] = "external_non_toxic"


print(
    "All non-ViToSA samples reset to "
    "'external_non_toxic'."
)


# ============================================================
# 20. SOURCE-BALANCED + TOXICITY-BALANCED SPEAKER ASSIGNMENT
# ============================================================
# Strategy:
#
# 1. ViToSA remains completely unchanged.
#
# 2. ALL FULL SOURCES are assigned jointly, not source-by-source.
#    This is important because one pseudo speaker may occur in
#    multiple FULL SOURCES.
#
# 3. A pseudo speaker is an atomic group:
#       one speaker -> exactly one split.
#
# 4. Assignment minimizes a combined objective:
#       - source balance
#       - toxicity balance
#       - total train/validation balance
#
#    The target train ratio is derived from the original ViToSA
#    train/validation ratio. This avoids imposing an arbitrary 50/50
#    split when ViToSA already has a different protocol.
#
# 5. FULL SOURCE rows are all used whenever a valid speaker embedding
#    exists. No FULL SOURCE row is intentionally discarded because
#    of toxicity or source composition.
#
# 6. FOSD remains a compensation source. It is considered only after
#    FULL SOURCE assignment and only for speakers with non-toxic > toxic.
#
# ============================================================

speaker_assignment = {}


# ------------------------------------------------------------
# Target train/validation ratio from original ViToSA
# ------------------------------------------------------------

vitosa_train_count = int(
    (
        (df["audio_source"] == BASE_SOURCE)
        & (df["split"] == "train")
    ).sum()
)

vitosa_val_count = int(
    (
        (df["audio_source"] == BASE_SOURCE)
        & (df["split"] == "validation")
    ).sum()
)

vitosa_train_val_total = (
    vitosa_train_count + vitosa_val_count
)

if vitosa_train_val_total > 0:
    TARGET_TRAIN_FRACTION = (
        vitosa_train_count
        / vitosa_train_val_total
    )
else:
    TARGET_TRAIN_FRACTION = 0.8

TARGET_VAL_FRACTION = 1.0 - TARGET_TRAIN_FRACTION

# Toxicity target.
# 0.50 means the algorithm prefers approximately balanced toxic /
# non-toxic data while respecting speaker atomicity and source usage.
TARGET_TOXIC_RATIO = 0.50

# Relative weights of the assignment objective.
SOURCE_BALANCE_WEIGHT = 3.0
TOXICITY_BALANCE_WEIGHT = 4.0
TOTAL_BALANCE_WEIGHT = 1.0


print("\n" + "=" * 80)
print("SOURCE-BALANCED + TOXICITY-BALANCED SPEAKER SPLIT")
print("=" * 80)
print(
    f"Target train fraction from ViToSA: "
    f"{TARGET_TRAIN_FRACTION:.4f}"
)
print(
    f"Target validation fraction: "
    f"{TARGET_VAL_FRACTION:.4f}"
)
print(
    f"Target toxic ratio: "
    f"{TARGET_TOXIC_RATIO:.4f}"
)


# ------------------------------------------------------------
# Helper: assign all rows of a speaker across FULL SOURCES
# ------------------------------------------------------------

def assign_full_speaker(
    speaker_id,
    split_name,
):
    """Assign every FULL SOURCE row belonging to one speaker."""

    mask = (
        embedding_df["pseudo_speaker_id"] == speaker_id
    ) & (
        embedding_df["audio_source"].isin(FULL_SOURCES)
    )

    indices = (
        embedding_df.loc[
            mask,
            "df_index",
        ]
        .astype(int)
        .tolist()
    )

    if not indices:
        return 0

    df.loc[
        indices,
        "split",
    ] = split_name

    return len(indices)


# ------------------------------------------------------------
# Build GLOBAL speaker groups for all FULL SOURCES
# ------------------------------------------------------------

full_rows = embedding_df[
    embedding_df["audio_source"].isin(FULL_SOURCES)
].copy()


if full_rows.empty:

    print(
        "[WARNING] No processed FULL SOURCE rows."
    )

else:

    full_speaker_stats = (
        full_rows
        .groupby("pseudo_speaker_id")
        .agg(
            total=("df_index", "count"),
            toxic=("toxicity", "sum"),
            non_toxic=(
                "toxicity",
                lambda x: int((x == 0).sum()),
            ),
            source_count=(
                "audio_source",
                "nunique",
            ),
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # Per-source totals
    # --------------------------------------------------------

    full_source_totals = (
        full_rows
        .groupby("audio_source")
        .size()
        .to_dict()
    )

    full_source_train_targets = {
        source: int(round(
            total * TARGET_TRAIN_FRACTION
        ))
        for source, total
        in full_source_totals.items()
    }

    full_source_val_targets = {
        source: (
            full_source_totals[source]
            - full_source_train_targets[source]
        )
        for source in full_source_totals
    }

    # --------------------------------------------------------
    # Global target counts
    # --------------------------------------------------------

    full_total = int(
        full_rows.shape[0]
    )

    target_full_train_total = int(round(
        full_total * TARGET_TRAIN_FRACTION
    ))

    target_full_val_total = (
        full_total
        - target_full_train_total
    )

    # --------------------------------------------------------
    # Speaker groups can contain multiple sources.
    # Larger / multi-source groups are assigned first so that
    # source diversity is decided early and not left to the tail.
    # --------------------------------------------------------

    full_speaker_stats = (
        full_speaker_stats
        .sort_values(
            by=[
                "source_count",
                "total",
                "non_toxic",
                "toxic",
            ],
            ascending=[
                False,
                False,
                False,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    train_source_counts = {
        source: 0
        for source in full_source_totals
    }
    val_source_counts = {
        source: 0
        for source in full_source_totals
    }

    train_toxic = 0
    train_non_toxic = 0
    val_toxic = 0
    val_non_toxic = 0
    train_total = 0
    val_total = 0


    def speaker_source_counts(speaker_id):
        rows = full_rows[
            full_rows["pseudo_speaker_id"]
            == speaker_id
        ]

        return (
            rows["audio_source"]
            .value_counts()
            .to_dict()
        )


    def split_objective(
        split_name,
        source_counts_after,
        total_after,
        toxic_after,
        non_toxic_after,
    ):
        """Lower is better."""

        if split_name == "train":
            split_fraction = TARGET_TRAIN_FRACTION
            target_total = target_full_train_total
        else:
            split_fraction = TARGET_VAL_FRACTION
            target_total = target_full_val_total

        # ----------------------------------------------------
        # A. Source balance
        # Compare each source's current fraction with its target
        # fraction inside the selected split.
        # ----------------------------------------------------

        source_error = 0.0

        for source, total in full_source_totals.items():
            if total <= 0:
                continue

            expected = (
                total
                * split_fraction
            )

            actual = source_counts_after.get(
                source,
                0,
            )

            source_error += (
                abs(actual - expected)
                / max(total, 1)
            )

        # ----------------------------------------------------
        # B. Toxicity balance
        # ----------------------------------------------------

        current_total = (
            toxic_after
            + non_toxic_after
        )

        if current_total > 0:
            current_toxic_ratio = (
                toxic_after
                / current_total
            )
        else:
            current_toxic_ratio = 0.5

        toxicity_error = abs(
            current_toxic_ratio
            - TARGET_TOXIC_RATIO
        )

        # ----------------------------------------------------
        # C. Total train/validation balance
        # ----------------------------------------------------

        total_error = abs(
            total_after - target_total
        ) / max(
            full_total,
            1,
        )

        return (
            SOURCE_BALANCE_WEIGHT
            * source_error
            + TOXICITY_BALANCE_WEIGHT
            * toxicity_error
            + TOTAL_BALANCE_WEIGHT
            * total_error
        )


    # ------------------------------------------------------------
    # Greedy multi-objective assignment
    # ------------------------------------------------------------

    for _, speaker_row in full_speaker_stats.iterrows():

        speaker_id = speaker_row[
            "pseudo_speaker_id"
        ]

        speaker_total = int(
            speaker_row["total"]
        )
        speaker_toxic = int(
            speaker_row["toxic"]
        )
        speaker_non_toxic = int(
            speaker_row["non_toxic"]
        )

        source_counts = speaker_source_counts(
            speaker_id
        )

        # ----------------------------------------------
        # Evaluate TRAIN
        # ----------------------------------------------

        train_source_after = {
            source: train_source_counts[source]
            for source in train_source_counts
        }

        for source, count in source_counts.items():
            train_source_after[source] += count

        train_score = split_objective(
            "train",
            train_source_after,
            train_total + speaker_total,
            train_toxic + speaker_toxic,
            train_non_toxic + speaker_non_toxic,
        )

        # ----------------------------------------------
        # Evaluate VALIDATION
        # ----------------------------------------------

        val_source_after = {
            source: val_source_counts[source]
            for source in val_source_counts
        }

        for source, count in source_counts.items():
            val_source_after[source] += count

        val_score = split_objective(
            "validation",
            val_source_after,
            val_total + speaker_total,
            val_toxic + speaker_toxic,
            val_non_toxic + speaker_non_toxic,
        )

        # ----------------------------------------------
        # Tie breaker: lower normalized total error
        # ----------------------------------------------

        if abs(train_score - val_score) < 1e-12:

            train_total_error = abs(
                (train_total + speaker_total)
                - target_full_train_total
            )

            val_total_error = abs(
                (val_total + speaker_total)
                - target_full_val_total
            )

            selected_split = (
                "train"
                if train_total_error
                <= val_total_error
                else "validation"
            )

        else:

            selected_split = (
                "train"
                if train_score < val_score
                else "validation"
            )

        # ----------------------------------------------
        # Commit assignment
        # ----------------------------------------------

        speaker_assignment[
            speaker_id
        ] = selected_split

        assigned = assign_full_speaker(
            speaker_id,
            selected_split,
        )

        if selected_split == "train":

            for source, count in source_counts.items():
                train_source_counts[source] += count

            train_total += speaker_total
            train_toxic += speaker_toxic
            train_non_toxic += speaker_non_toxic

        else:

            for source, count in source_counts.items():
                val_source_counts[source] += count

            val_total += speaker_total
            val_toxic += speaker_toxic
            val_non_toxic += speaker_non_toxic


    print("\nFULL SOURCE ASSIGNMENT")
    print("-" * 80)
    print(
        f"Speakers assigned : "
        f"{len(full_speaker_stats):,}"
    )
    print(
        f"Train speakers    : "
        f"{sum(1 for s in full_speaker_stats['pseudo_speaker_id'] if speaker_assignment.get(s) == 'train'):,}"
    )
    print(
        f"Validation speakers: "
        f"{sum(1 for s in full_speaker_stats['pseudo_speaker_id'] if speaker_assignment.get(s) == 'validation'):,}"
    )

    print("\nSOURCE BALANCE")
    print("-" * 80)

    for source in sorted(full_source_totals):
        total = full_source_totals[source]
        train_count = train_source_counts[source]
        val_count = val_source_counts[source]

        print(
            f"{source:25s} | "
            f"total={total:7,} | "
            f"train={train_count:7,} "
            f"({train_count / max(total, 1):.3f}) | "
            f"validation={val_count:7,} "
            f"({val_count / max(total, 1):.3f})"
        )

    print_balance(
        "BALANCE AFTER SOURCE + TOXICITY ASSIGNMENT"
    )


# ============================================================
# 21. FOSD COMPENSATION
# ============================================================

print("\n" + "=" * 80)
print("FOSD - TOXICITY COMPENSATION ONLY")
print("=" * 80)

fosd_rows = embedding_df[
    embedding_df["audio_source"] == FOSD_SOURCE
].copy()


if fosd_rows.empty:

    print(
        "[WARNING] No processed FOSD samples."
    )

else:

    print(
        f"FOSD processed samples: "
        f"{len(fosd_rows):,}"
    )

    fosd_speakers = (
        fosd_rows[
            "pseudo_speaker_id"
        ]
        .dropna()
        .unique()
        .tolist()
    )

    # A FOSD speaker already used by a FULL SOURCE is locked to
    # that split and must not be reused independently.
    fosd_candidate_speakers = [
        speaker_id
        for speaker_id in fosd_speakers
        if speaker_id
        not in speaker_assignment
    ]

    fosd_candidates = (
        fosd_rows[
            fosd_rows[
                "pseudo_speaker_id"
            ].isin(
                fosd_candidate_speakers
            )
        ]
        .groupby(
            "pseudo_speaker_id"
        )
        .agg(
            total=(
                "df_index",
                "count"
            ),
            toxic=(
                "toxicity",
                "sum"
            ),
            non_toxic=(
                "toxicity",
                lambda x: int(
                    (x == 0).sum()
                )
            ),
        )
        .reset_index()
    )

    fosd_candidates["delta"] = (
        fosd_candidates["non_toxic"]
        - fosd_candidates["toxic"]
    )

    # Only speakers that improve the non-toxic side are eligible.
    fosd_candidates = (
        fosd_candidates[
            fosd_candidates["delta"] > 0
        ]
        .sort_values(
            by=[
                "delta",
                "non_toxic",
                "total",
            ],
            ascending=[
                False,
                False,
                True,
            ]
        )
        .reset_index(drop=True)
    )

    print(
        f"FOSD eligible speakers: "
        f"{len(fosd_candidates):,}"
    )


    def apply_fosd_compensation(
        split_name,
    ):
        """Add FOSD speakers until toxic/non-toxic is balanced."""

        toxic, non_toxic, _ = get_split_counts(
            split_name
        )

        deficit = max(
            0,
            toxic - non_toxic,
        )

        print(
            f"\n{split_name.upper()} "
            f"non-toxic deficit before FOSD: "
            f"{deficit:,}"
        )

        if deficit <= 0:
            return

        gained = 0

        for _, row in fosd_candidates.iterrows():

            if gained >= deficit:
                break

            speaker_id = row[
                "pseudo_speaker_id"
            ]

            if speaker_id in speaker_assignment:
                continue

            speaker_toxic = int(
                row["toxic"]
            )
            speaker_non_toxic = int(
                row["non_toxic"]
            )
            delta = (
                speaker_non_toxic
                - speaker_toxic
            )

            speaker_assignment[
                speaker_id
            ] = split_name

            assigned = assign_source_speaker(
                FOSD_SOURCE,
                speaker_id,
                split_name,
            )

            gained += delta

            print(
                f"[FOSD -> {split_name}] "
                f"{speaker_id} | "
                f"samples={assigned:,} | "
                f"toxic={speaker_toxic:,} | "
                f"non-toxic={speaker_non_toxic:,} | "
                f"delta={delta:+,}"
            )


    apply_fosd_compensation("train")
    apply_fosd_compensation("validation")


# ============================================================
# 22. ALL REMAINING FOSD -> EXTERNAL
# ============================================================

if not fosd_rows.empty:

    fosd_indices = set(
        fosd_rows[
            "df_index"
        ]
        .astype(int)
        .tolist()
    )

    fosd_train_val_indices = set(
        df.loc[
            (
                df["audio_source"]
                == FOSD_SOURCE
            )
            & (
                df["split"].isin(
                    [
                        "train",
                        "validation",
                    ]
                )
            )
        ]
        .index
        .astype(int)
        .tolist()
    )

    remaining_fosd_indices = (
        fosd_indices
        - fosd_train_val_indices
    )

    if remaining_fosd_indices:

        df.loc[
            list(remaining_fosd_indices),
            "split"
        ] = "external_non_toxic"

    print(
        "\nRemaining FOSD reserve: "
        f"{len(remaining_fosd_indices):,}"
    )


# ============================================================
# 23. SOURCE-BALANCE SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("FINAL SOURCE BALANCE - TRAIN / VALIDATION")
print("=" * 80)

for source in FULL_SOURCES:

    source_total = int(
        (
            df["audio_source"]
            == source
        ).sum()
    )

    source_train = int(
        (
            (df["audio_source"] == source)
            & (df["split"] == "train")
        ).sum()
    )

    source_val = int(
        (
            (df["audio_source"] == source)
            & (df["split"] == "validation")
        ).sum()
    )

    source_external = int(
        (
            (df["audio_source"] == source)
            & (
                df["split"]
                == "external_non_toxic"
            )
        ).sum()
    )

    print(
        f"{source:25s} | "
        f"total={source_total:7,} | "
        f"train={source_train:7,} "
        f"({source_train / max(source_total, 1):.3f}) | "
        f"validation={source_val:7,} "
        f"({source_val / max(source_total, 1):.3f}) | "
        f"external={source_external:7,}"
    )


# ============================================================
# 24. FINAL TOXICITY BALANCE SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("FINAL TOXICITY BALANCE")
print("=" * 80)

for split_name in [
    "train",
    "validation",
]:

    toxic, non_toxic, total = get_split_counts(
        split_name
    )

    print(
        f"{split_name:12s} | "
        f"total={total:,} | "
        f"toxic={toxic:,} | "
        f"non-toxic={non_toxic:,} | "
        f"non-toxic/toxic="
        f"{non_toxic / max(toxic, 1):.3f}"
    )

# 25. VERIFY ViToSA WAS NOT CHANGED
# ============================================================

print("\n" + "=" * 80)
print("VERIFY VITOSA SPLIT")
print("=" * 80)


original_vitosa = (
    pd.read_csv(INPUT_CSV)
    .reset_index(drop=True)
)


original_vitosa_mask = (
    original_vitosa[
        "audio_source"
    ]
    == BASE_SOURCE
)


current_vitosa_mask = (
    df[
        "audio_source"
    ]
    == BASE_SOURCE
)


original_vitosa_splits = (
    original_vitosa.loc[
        original_vitosa_mask,
        "split"
    ]
    .reset_index(drop=True)
)


current_vitosa_splits = (
    df.loc[
        current_vitosa_mask,
        "split"
    ]
    .reset_index(drop=True)
)


if original_vitosa_splits.equals(
    current_vitosa_splits
):

    print(
        "OK: ViToSA split unchanged."
    )

else:

    print(
        "WARNING: ViToSA split changed!"
    )


# ============================================================
# 26. FINAL BALANCE
# ============================================================

print_balance(
    "FINAL TRAIN / VALIDATION BALANCE"
)


# ============================================================
# 27. FINAL SOURCE × SPLIT
# ============================================================

print("\n" + "=" * 80)
print("FINAL SOURCE × SPLIT")
print("=" * 80)


source_split = pd.crosstab(
    df["audio_source"],
    df["split"],
    margins=True,
    margins_name="Tổng_Cộng"
)


print(
    source_split
)


# ============================================================
# 28. FINAL SOURCE × TOXICITY
# ============================================================

print("\n" + "=" * 80)
print("FINAL SOURCE × TOXICITY")
print("=" * 80)


source_toxicity = pd.crosstab(
    df["audio_source"],
    df["toxicity"],
    margins=True,
    margins_name="Tổng_Cộng"
)


print(
    source_toxicity
)


# ============================================================
# 29. FINAL SPLIT × TOXICITY
# ============================================================

print("\n" + "=" * 80)
print("FINAL SPLIT × TOXICITY")
print("=" * 80)


split_toxicity = pd.crosstab(
    df["split"],
    df["toxicity"],
    margins=True,
    margins_name="Tổng_Cộng"
)


print(
    split_toxicity
)


# ============================================================
# 30. SOURCE × SPLIT × TOXICITY
# ============================================================

print("\n" + "=" * 80)
print("SOURCE × SPLIT × TOXICITY")
print("=" * 80)


full_crosstab = pd.crosstab(
    [
        df["audio_source"],
        df["split"],
    ],
    df["toxicity"],
    margins=True,
    margins_name="Tổng_Cộng"
)


print(
    full_crosstab
)


# ============================================================
# 31. SPEAKER INDEPENDENCE CHECK
# ============================================================

train_speaker_ids = set(
    df.loc[
        df["split"] == "train",
        "pseudo_speaker_id"
    ]
    .dropna()
    .astype(str)
)


val_speaker_ids = set(
    df.loc[
        df["split"] == "validation",
        "pseudo_speaker_id"
    ]
    .dropna()
    .astype(str)
)


speaker_overlap = (
    train_speaker_ids
    &
    val_speaker_ids
)


print("\n" + "=" * 80)
print("SPEAKER INDEPENDENCE CHECK")
print("=" * 80)


print(
    f"Train speakers      : "
    f"{len(train_speaker_ids):,}"
)


print(
    f"Validation speakers : "
    f"{len(val_speaker_ids):,}"
)


print(
    f"Speaker overlap     : "
    f"{len(speaker_overlap):,}"
)


if speaker_overlap:

    print(
        "❌ WARNING: "
        "Speaker leakage detected!"
    )

    print(
        "\nOverlapping speakers:"
    )

    for speaker_id in sorted(
        speaker_overlap
    )[:50]:

        print(
            f"  {speaker_id}"
        )

else:

    print(
        "✅ No speaker overlap between "
        "train and validation."
    )


# ============================================================
# 32. SPEAKER MULTI-SPLIT CHECK
# ============================================================

print("\n" + "=" * 80)
print("SPEAKER MULTI-SPLIT CHECK")
print("=" * 80)


speaker_split_counts = (
    df[
        df["pseudo_speaker_id"]
        .notna()
    ]
    .groupby(
        "pseudo_speaker_id"
    )["split"]
    .nunique()
)


multi_split_speakers = (
    speaker_split_counts[
        speaker_split_counts > 1
    ]
)


print(
    f"Speakers with >1 split: "
    f"{len(multi_split_speakers):,}"
)


if len(multi_split_speakers) == 0:

    print(
        "✅ Every pseudo speaker belongs "
        "to only one split."
    )

else:

    print(
        "❌ WARNING: Some speakers occur "
        "in multiple splits."
    )


# ============================================================
# 33. FULL SOURCE USAGE CHECK
# ============================================================

print("\n" + "=" * 80)
print("FULL SOURCE USAGE CHECK")
print("=" * 80)


for source in FULL_SOURCES:

    total_count = int(
        (
            df["audio_source"]
            == source
        ).sum()
    )


    train_val_count = int(
        (
            (
                df["audio_source"]
                == source
            )
            &
            (
                df["split"].isin(
                    [
                        "train",
                        "validation",
                    ]
                )
            )
        ).sum()
    )


    external_count = int(
        (
            (
                df["audio_source"]
                == source
            )
            &
            (
                df["split"]
                == "external_non_toxic"
            )
        ).sum()
    )


    print(
        f"{source:25s} | "
        f"total={total_count:7,} | "
        f"train/val={train_val_count:7,} | "
        f"external={external_count:7,}"
    )


    if train_val_count == total_count:

        print(
            " " * 4
            + "✅ FULL SOURCE completely used."
        )

    else:

        print(
            " " * 4
            + "⚠️ FULL SOURCE has unused rows."
        )


# ============================================================
# 34. FOSD USAGE CHECK
# ============================================================

if FOSD_SOURCE in available_sources:

    print("\n" + "=" * 80)
    print("FOSD USAGE CHECK")
    print("=" * 80)


    fosd_total = int(
        (
            df["audio_source"]
            == FOSD_SOURCE
        ).sum()
    )


    fosd_train = int(
        (
            (
                df["audio_source"]
                == FOSD_SOURCE
            )
            &
            (
                df["split"]
                == "train"
            )
        ).sum()
    )


    fosd_val = int(
        (
            (
                df["audio_source"]
                == FOSD_SOURCE
            )
            &
            (
                df["split"]
                == "validation"
            )
        ).sum()
    )


    fosd_external = int(
        (
            (
                df["audio_source"]
                == FOSD_SOURCE
            )
            &
            (
                df["split"]
                == "external_non_toxic"
            )
        ).sum()
    )


    fosd_other = int(
        (
            (
                df["audio_source"]
                == FOSD_SOURCE
            )
            &
            (
                ~df["split"].isin(
                    [
                        "train",
                        "validation",
                        "external_non_toxic",
                    ]
                )
            )
        ).sum()
    )


    print(
        f"FOSD total             : "
        f"{fosd_total:,}"
    )


    print(
        f"FOSD -> train          : "
        f"{fosd_train:,}"
    )


    print(
        f"FOSD -> validation     : "
        f"{fosd_val:,}"
    )


    print(
        f"FOSD -> external       : "
        f"{fosd_external:,}"
    )


    print(
        f"FOSD -> other          : "
        f"{fosd_other:,}"
    )


    print(
        f"Check total            : "
        f"{fosd_train + fosd_val + fosd_external + fosd_other:,}"
    )


    if (
        fosd_train
        + fosd_val
        + fosd_external
        + fosd_other
        == fosd_total
    ):

        print(
            "✅ FOSD accounting is consistent."
        )

    else:

        print(
            "❌ FOSD accounting mismatch!"
        )


# ============================================================
# 35. CHECK TRAIN / VALIDATION BALANCE REQUIREMENT
# ============================================================

print("\n" + "=" * 80)
print("NON-TOXIC BALANCE REQUIREMENT")
print("=" * 80)


for split_name in [
    "train",
    "validation",
]:

    toxic, non_toxic, total = (
        get_split_counts(
            split_name
        )
    )


    print(
        f"{split_name:12s} | "
        f"toxic={toxic:,} | "
        f"non-toxic={non_toxic:,} | "
        f"non-toxic >= toxic: "
        f"{non_toxic >= toxic}"
    )


# ============================================================
# 36. FINAL SPLIT DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("FINAL SPLIT DISTRIBUTION")
print("=" * 80)


print(
    df["split"]
    .value_counts(
        dropna=False
    )
)


# ============================================================
# 37. CHECK MISSING SPLIT
# ============================================================

missing_split_count = int(
    df["split"]
    .isna()
    .sum()
)


print(
    f"\nMissing split: "
    f"{missing_split_count:,}"
)


if missing_split_count == 0:

    print(
        "✅ No missing split."
    )

else:

    print(
        "❌ WARNING: "
        "Some rows have missing split."
    )


# ============================================================
# 38. CHECK pseudo_speaker_id
# ============================================================

missing_speaker_count = int(
    df[
        df["audio_source"]
        != BASE_SOURCE
    ][
        "pseudo_speaker_id"
    ]
    .isna()
    .sum()
)


print(
    f"Non-ViToSA rows without "
    f"pseudo speaker: "
    f"{missing_speaker_count:,}"
)


# ============================================================
# 39. SAVE
# ============================================================

print("\n" + "=" * 80)
print("SAVING")
print("=" * 80)


df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"Output:\n"
    f"{OUTPUT_CSV}"
)


print(
    f"Total rows: "
    f"{len(df):,}"
)


# ============================================================
# 40. FINAL SUCCESS MESSAGE
# ============================================================

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)


if len(speaker_overlap) == 0:

    print(
        "✅ Speaker independence: PASS"
    )

else:

    print(
        "❌ Speaker independence: FAIL"
    )


if missing_split_count == 0:

    print(
        "✅ Split completeness: PASS"
    )

else:

    print(
        "❌ Split completeness: FAIL"
    )


print(
    "\nFinal dataset saved successfully."
)

a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA SpeechRun\envLab\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE
Device: cuda
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU

LOADING DATASET
Total samples: 32,303

AUDIO SOURCE
audio_source
vitosa          12802
fosd            12600
retrieval        4542
common_voice     2359
Name: count, dtype: int64

ORIGINAL SPLIT
split
external_non_toxic    14143
train                 13183
validation             2161
test                   2000
NaN                     816
Name: count, dtype: int64

SOURCE × TOXICITY
toxicity          0      1  Tổng_Cộng
audio_source                         
common_voice   2073    286       2359
fosd          12070    530      12600
retrieval      4542      0       4542
vitosa         1000  11802      12802
Tổng_Cộng     19685  12618      32303

SOURCE CONFIGURATION
Base source: vitosa

FULL SOURCES:
  01. common_voice
  02. retrieval

COMPENSATION SOURCE:
  fosd

TARGET DATA FOR SPEAKER CLUSTERING
Target samples: 19,501
audio_source
fosd            12600
retrieval        4542
common_voice     2359
Name: c

a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA SpeechRun\envLab\Lib\site-packages\speechbrain\utils\checkpoints.py:202: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  stat

ECAPA-TDNN loaded successfully.

EXTRACTING SPEAKER EMBEDDINGS


ECAPA: 100%|██████████| 19501/19501 [08:58<00:00, 36.23it/s]



EMBEDDING EXTRACTION COMPLETED
Valid embeddings : 19,457
Skipped audio    : 44

Skipped report saved:
A:\A _ Working\Researching\B - AIoT Lab VN\VITOSA SpeechRun\vitosa_datasets\speaker_split_skipped_audio5.1.csv

Embedding shape:
(19457, 192)

CLUSTERING SPEAKERS
Detected pseudo speakers: 5,484

SPEAKER SUMMARY
   pseudo_speaker_id  samples       sources  toxic  non_toxic  delta
0      SPEAKER_00000        8          fosd      0          8      8
1      SPEAKER_00001       12     retrieval      0         12     12
2      SPEAKER_00002        2     retrieval      0          2      2
3      SPEAKER_00003       11          fosd      0         11     11
4      SPEAKER_00004       16          fosd      1         15     14
5      SPEAKER_00005        4     retrieval      0          4      4
6      SPEAKER_00006        8     retrieval      0          8      8
7      SPEAKER_00007       19          fosd      1         18     17
8      SPEAKER_00008        2          fosd      0          2   

# refix 5.1.2

In [1]:
# ============================================================
# SPEAKER-INDEPENDENT DATASET SPLIT
# ============================================================
#
# LOGIC
#
# 1. ViToSA
#    -> GIỮ NGUYÊN train / validation / test
#
# 2. FULL SOURCES
#    -> Common Voice
#    -> Retrieval
#
#    -> DÙNG TOÀN BỘ
#    -> Speaker là atomic group
#    -> Speaker đã được assign split nào
#       thì source khác của cùng speaker cũng theo split đó
#
# 3. FOSD
#    -> CHỈ dùng để compensation non-toxic
#    -> Không tự động được kéo vào train/validation
#       chỉ vì trùng speaker với FULL SOURCE
#
#    -> Nếu train:
#          non-toxic < toxic
#       thì lấy FOSD speaker có lợi thế non-toxic
#
#    -> Nếu validation:
#          non-toxic < toxic
#       thì tiếp tục lấy FOSD speaker
#
#    -> FOSD còn dư:
#          external_non_toxic
#
# 4. IMPORTANT
#
#    Speaker là atomic group.
#
#    Không được có:
#
#        speaker A -> train
#        speaker A -> validation
#
#    Final requirement:
#
#        train_speakers ∩ validation_speakers = ∅
#
# ============================================================


# ============================================================
# 0. IMPORT
# ============================================================

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import soundfile as sf

from tqdm.auto import tqdm

from speechbrain.inference.speaker import EncoderClassifier

from sklearn.cluster import AgglomerativeClustering


# ============================================================
# 1. CONFIG
# ============================================================

# ------------------------------------------------------------
# INPUT CSV
# ------------------------------------------------------------

INPUT_CSV = Path(
    r"A:\A _ Working\Researching\B - AIoT Lab VN\VITOSA SpeechRun\vitosa_datasets\final_vietnamese_toxic_utterance_dataset_v5.1.csv"
)


# ------------------------------------------------------------
# AUDIO FOLDER
# ------------------------------------------------------------

WAV_FOLDER = Path(
    r"A:\A _ Working\Researching\B - AIoT Lab VN\VITOSA SpeechRun\vitosa_datasets\wav_segments_v5.1"
)


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_CSV = INPUT_CSV.parent / (
    "final_vietnamese_toxic_utterance_dataset_v5.1.2_speaker_split.csv"
)


# ------------------------------------------------------------
# SKIPPED AUDIO REPORT
# ------------------------------------------------------------

SKIPPED_OUTPUT_CSV = INPUT_CSV.parent / (
    "speaker_split_skipped_audio5.1.2.csv"
)


# ------------------------------------------------------------
# RANDOM SEED
# ------------------------------------------------------------

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)


# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

device_str = (
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)


# ------------------------------------------------------------
# ECAPA CONFIG
# ------------------------------------------------------------

ECAPA_SOURCE = "speechbrain/spkrec-ecapa-voxceleb"


# ------------------------------------------------------------
# SPEAKER CLUSTERING
# ------------------------------------------------------------

SPEAKER_DISTANCE_THRESHOLD = 0.35


# ------------------------------------------------------------
# REQUIRED SAMPLE RATE
# ------------------------------------------------------------

TARGET_SAMPLE_RATE = 16000


# ------------------------------------------------------------
# MINIMUM AUDIO LENGTH
# ------------------------------------------------------------

MIN_AUDIO_SECONDS = 1.0

MIN_AUDIO_SAMPLES = int(
    TARGET_SAMPLE_RATE * MIN_AUDIO_SECONDS
)


# ============================================================
# 2. PRINT DEVICE
# ============================================================

print("=" * 80)
print("DEVICE")
print("=" * 80)

print(f"Device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():

    print(
        f"GPU: "
        f"{torch.cuda.get_device_name(0)}"
    )


# ============================================================
# 3. LOAD DATASET
# ============================================================

print("\n" + "=" * 80)
print("LOADING DATASET")
print("=" * 80)

if not INPUT_CSV.exists():

    raise FileNotFoundError(
        f"INPUT_CSV không tồn tại:\n{INPUT_CSV}"
    )


df = pd.read_csv(INPUT_CSV)

# ------------------------------------------------------------
# Reset pandas index
# ------------------------------------------------------------

df = df.reset_index(drop=True)


# ------------------------------------------------------------
# Create stable row ID
# ------------------------------------------------------------

df["__row_id"] = np.arange(
    len(df),
    dtype=np.int64
)


# ------------------------------------------------------------
# Required columns
# ------------------------------------------------------------

required_columns = [
    "audio_path",
    "audio_source",
    "toxicity",
    "split",
]


missing_columns = [
    col
    for col in required_columns
    if col not in df.columns
]


if missing_columns:

    raise ValueError(
        "Dataset thiếu các column bắt buộc:\n"
        + "\n".join(
            f"- {col}"
            for col in missing_columns
        )
    )


print(
    f"Total samples: {len(df):,}"
)


# ============================================================
# 4. DATASET SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("AUDIO SOURCE")
print("=" * 80)

print(
    df["audio_source"]
    .value_counts(dropna=False)
)


print("\n" + "=" * 80)
print("ORIGINAL SPLIT")
print("=" * 80)

print(
    df["split"]
    .value_counts(dropna=False)
)


print("\n" + "=" * 80)
print("SOURCE × TOXICITY")
print("=" * 80)

print(
    pd.crosstab(
        df["audio_source"],
        df["toxicity"],
        margins=True,
        margins_name="Tổng_Cộng"
    )
)


# ============================================================
# 5. CREATE pseudo_speaker_id
# ============================================================

df["pseudo_speaker_id"] = pd.NA


# ============================================================
# 6. SOURCE CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# BASE SOURCE
# ------------------------------------------------------------

BASE_SOURCE = "vitosa"


# ------------------------------------------------------------
# FULL SOURCES
# ------------------------------------------------------------

FULL_SOURCE_ORDER = [
    "common_voice",
    "retrieval",
]


# ------------------------------------------------------------
# COMPENSATION SOURCE
# ------------------------------------------------------------

FOSD_SOURCE = "fosd"


# ------------------------------------------------------------
# Available sources
# ------------------------------------------------------------

available_sources = (
    df["audio_source"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)


# ------------------------------------------------------------
# FULL SOURCES that exist
# ------------------------------------------------------------

FULL_SOURCES = [
    source
    for source in FULL_SOURCE_ORDER
    if source in available_sources
]


# ------------------------------------------------------------
# Unknown sources
#
# Any source other than:
#
#     ViToSA
#     FOSD
#
# is considered FULL SOURCE
# ------------------------------------------------------------

UNKNOWN_FULL_SOURCES = [
    source
    for source in available_sources
    if (
        source != BASE_SOURCE
        and source != FOSD_SOURCE
        and source not in FULL_SOURCES
    )
]


FULL_SOURCES.extend(
    UNKNOWN_FULL_SOURCES
)


# ------------------------------------------------------------
# Target sources for ECAPA
# ------------------------------------------------------------

TARGET_SOURCES = (
    FULL_SOURCES
    + (
        [FOSD_SOURCE]
        if FOSD_SOURCE in available_sources
        else []
    )
)


print("\n" + "=" * 80)
print("SOURCE CONFIGURATION")
print("=" * 80)

print(
    f"Base source: "
    f"{BASE_SOURCE}"
)

print("\nFULL SOURCES:")

for i, source in enumerate(
    FULL_SOURCES,
    start=1
):

    print(
        f"  {i:02d}. {source}"
    )


print("\nCOMPENSATION SOURCE:")

print(
    f"  {FOSD_SOURCE}"
)


# ============================================================
# 7. TARGET DATA FOR SPEAKER CLUSTERING
# ============================================================

target_df = df[
    df["audio_source"].isin(
        TARGET_SOURCES
    )
].copy()


print("\n" + "=" * 80)
print("TARGET DATA FOR SPEAKER CLUSTERING")
print("=" * 80)

print(
    f"Target samples: "
    f"{len(target_df):,}"
)


print(
    target_df["audio_source"]
    .value_counts()
)


# ============================================================
# 8. BUILD WAV INDEX
# ============================================================

print("\n" + "=" * 80)
print("BUILDING WAV INDEX")
print("=" * 80)


if not WAV_FOLDER.exists():

    raise FileNotFoundError(
        f"WAV_FOLDER không tồn tại:\n{WAV_FOLDER}"
    )


print(
    f"Scanning:\n{WAV_FOLDER}"
)


wav_files = list(
    WAV_FOLDER.rglob("*.wav")
)


wav_index = {}


for wav_path in wav_files:

    filename = wav_path.name.lower()

    if filename not in wav_index:

        wav_index[filename] = wav_path


print(
    f"Found WAV files: "
    f"{len(wav_files):,}"
)


# ============================================================
# 9. LOAD ECAPA-TDNN
# ============================================================

print("\n" + "=" * 80)
print("LOADING ECAPA-TDNN")
print("=" * 80)


classifier = EncoderClassifier.from_hparams(
    source=ECAPA_SOURCE,
    run_opts={
        "device": device_str
    }
)


print("ECAPA-TDNN loaded successfully.")


# ============================================================
# 10. EXTRACT SPEAKER EMBEDDINGS
# ============================================================

embeddings = []

metadata = []

skipped_audio = []


print("\n" + "=" * 80)
print("EXTRACTING SPEAKER EMBEDDINGS")
print("=" * 80)


for idx, row in tqdm(
    target_df.iterrows(),
    total=len(target_df),
    desc="ECAPA"
):

    # --------------------------------------------------------
    # Stable pandas index
    # --------------------------------------------------------

    df_index = int(idx)


    # --------------------------------------------------------
    # Audio filename
    # --------------------------------------------------------

    audio_filename = Path(
        str(row["audio_path"])
    ).name


    wav_path = wav_index.get(
        audio_filename.lower()
    )


    # --------------------------------------------------------
    # AUDIO NOT FOUND
    # --------------------------------------------------------

    if wav_path is None:

        skipped_audio.append(
            {
                "df_index": df_index,
                "audio_path": audio_filename,
                "audio_source": row[
                    "audio_source"
                ],
                "reason": "audio_not_found",
            }
        )

        continue


    try:

        # ----------------------------------------------------
        # LOAD WAV
        # ----------------------------------------------------

        waveform, sample_rate = sf.read(
            str(wav_path),
            dtype="float32"
        )


        # ----------------------------------------------------
        # MONO
        # ----------------------------------------------------

        if waveform.ndim > 1:

            waveform = waveform.mean(
                axis=1
            )


        # ----------------------------------------------------
        # SAMPLE RATE
        # ----------------------------------------------------

        if sample_rate != TARGET_SAMPLE_RATE:

            skipped_audio.append(
                {
                    "df_index": df_index,
                    "audio_path": audio_filename,
                    "audio_source": row[
                        "audio_source"
                    ],
                    "reason": (
                        f"invalid_sr_{sample_rate}"
                    ),
                }
            )

            continue


        # ----------------------------------------------------
        # TOO SHORT
        # ----------------------------------------------------

        if len(waveform) < MIN_AUDIO_SAMPLES:

            skipped_audio.append(
                {
                    "df_index": df_index,
                    "audio_path": audio_filename,
                    "audio_source": row[
                        "audio_source"
                    ],
                    "reason": "audio_too_short",
                }
            )

            continue


        # ----------------------------------------------------
        # TORCH
        # ----------------------------------------------------

        wav_tensor = torch.tensor(
            waveform,
            dtype=torch.float32
        )


        # ----------------------------------------------------
        # Add batch dimension
        # ----------------------------------------------------

        wav_input = (
            wav_tensor
            .unsqueeze(0)
            .to(device)
        )


        # ----------------------------------------------------
        # ECAPA
        # ----------------------------------------------------

        with torch.no_grad():

            embedding = (
                classifier
                .encode_batch(
                    wav_input
                )
                .squeeze()
                .detach()
                .cpu()
                .numpy()
            )


        # ----------------------------------------------------
        # Validate embedding
        # ----------------------------------------------------

        if embedding.ndim != 1:

            embedding = embedding.reshape(-1)


        if not np.isfinite(
            embedding
        ).all():

            skipped_audio.append(
                {
                    "df_index": df_index,
                    "audio_path": audio_filename,
                    "audio_source": row[
                        "audio_source"
                    ],
                    "reason": (
                        "invalid_embedding"
                    ),
                }
            )

            continue


        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------

        embeddings.append(
            embedding
        )


        metadata.append(
            {
                "df_index": df_index,
                "audio_path": audio_filename,
                "audio_source": row[
                    "audio_source"
                ],
                "toxicity": int(
                    row["toxicity"]
                ),
            }
        )


    except Exception as e:

        skipped_audio.append(
            {
                "df_index": df_index,
                "audio_path": audio_filename,
                "audio_source": row[
                    "audio_source"
                ],
                "reason": (
                    f"error_{type(e).__name__}: "
                    f"{str(e)}"
                ),
            }
        )


# ============================================================
# 11. EMBEDDING EXTRACTION SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("EMBEDDING EXTRACTION COMPLETED")
print("=" * 80)


print(
    f"Valid embeddings : "
    f"{len(embeddings):,}"
)


print(
    f"Skipped audio    : "
    f"{len(skipped_audio):,}"
)


# ------------------------------------------------------------
# Save skipped report
# ------------------------------------------------------------

if skipped_audio:

    skipped_df = pd.DataFrame(
        skipped_audio
    )

    skipped_df.to_csv(
        SKIPPED_OUTPUT_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"\nSkipped report saved:\n"
        f"{SKIPPED_OUTPUT_CSV}"
    )


if len(embeddings) == 0:

    raise RuntimeError(
        "Không có embedding nào được tạo."
    )


# ============================================================
# 12. CREATE EMBEDDING DATAFRAME
# ============================================================

X = np.asarray(
    embeddings,
    dtype=np.float32
)


embedding_df = pd.DataFrame(
    metadata
)


print("\nEmbedding shape:")

print(
    X.shape
)


# ============================================================
# 13. CLUSTER SPEAKERS
# ============================================================

print("\n" + "=" * 80)
print("CLUSTERING SPEAKERS")
print("=" * 80)


cluster_model = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=(
        SPEAKER_DISTANCE_THRESHOLD
    ),
    metric="cosine",
    linkage="average",
)


cluster_labels = (
    cluster_model
    .fit_predict(X)
)


embedding_df["cluster_id"] = (
    cluster_labels
)


print(
    f"Detected pseudo speakers: "
    f"{embedding_df['cluster_id'].nunique():,}"
)


# ============================================================
# 14. GLOBAL pseudo_speaker_id
# ============================================================

embedding_df[
    "pseudo_speaker_id"
] = (
    embedding_df[
        "cluster_id"
    ]
    .apply(
        lambda x:
        f"SPEAKER_{int(x):05d}"
    )
)


# ============================================================
# 15. WRITE pseudo_speaker_id BACK
# ============================================================

for _, row in embedding_df.iterrows():

    df.loc[
        int(row["df_index"]),
        "pseudo_speaker_id"
    ] = row[
        "pseudo_speaker_id"
    ]


# ============================================================
# 16. SPEAKER SUMMARY
# ============================================================

speaker_summary = (
    embedding_df
    .groupby(
        "pseudo_speaker_id"
    )
    .agg(
        samples=(
            "df_index",
            "count"
        ),
        sources=(
            "audio_source",
            lambda x:
            ",".join(
                sorted(
                    set(x)
                )
            )
        ),
        toxic=(
            "toxicity",
            "sum"
        ),
        non_toxic=(
            "toxicity",
            lambda x:
            int(
                (x == 0).sum()
            )
        ),
    )
    .reset_index()
)


speaker_summary["delta"] = (
    speaker_summary["non_toxic"]
    - speaker_summary["toxic"]
)


print("\n" + "=" * 80)
print("SPEAKER SUMMARY")
print("=" * 80)


print(
    speaker_summary.head(20)
)


# ============================================================
# 17. KEEP ORIGINAL VITOSA SPLITS
# ============================================================

vitosa_df = df[
    df["audio_source"]
    == BASE_SOURCE
].copy()


print("\n" + "=" * 80)
print("ORIGINAL VITOSA SPLIT")
print("=" * 80)


print(
    pd.crosstab(
        vitosa_df["split"],
        vitosa_df["toxicity"],
        margins=True,
        margins_name="Tổng_Cộng"
    )
)


# ============================================================
# 18. HELPER FUNCTIONS
# ============================================================

def get_split_counts(
    split_name
):

    subset = df[
        df["split"]
        == split_name
    ]

    toxic = int(
        (
            subset["toxicity"]
            == 1
        ).sum()
    )

    non_toxic = int(
        (
            subset["toxicity"]
            == 0
        ).sum()
    )

    total = (
        toxic
        + non_toxic
    )

    return (
        toxic,
        non_toxic,
        total
    )


def get_deficit(
    split_name
):

    toxic, non_toxic, _ = (
        get_split_counts(
            split_name
        )
    )

    return max(
        0,
        toxic - non_toxic
    )


def print_balance(
    title
):

    train_toxic, train_non_toxic, train_total = (
        get_split_counts("train")
    )

    val_toxic, val_non_toxic, val_total = (
        get_split_counts("validation")
    )


    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


    print(
        f"TRAIN       "
        f"total={train_total:,} | "
        f"toxic={train_toxic:,} | "
        f"non-toxic={train_non_toxic:,} | "
        f"non-toxic/toxic="
        f"{train_non_toxic / max(train_toxic, 1):.3f}"
    )


    print(
        f"VALIDATION  "
        f"total={val_total:,} | "
        f"toxic={val_toxic:,} | "
        f"non-toxic={val_non_toxic:,} | "
        f"non-toxic/toxic="
        f"{val_non_toxic / max(val_toxic, 1):.3f}"
    )


def assign_source_speaker(
    source,
    speaker_id,
    split_name
):
    """
    Chỉ assign ROWS thuộc source hiện tại
    của speaker hiện tại.

    Không kéo source khác theo.

    Đây là điểm quan trọng để FOSD
    không bị kéo vào train/validation
    chỉ vì speaker trùng với FULL SOURCE.
    """

    mask = (
        (
            embedding_df[
                "audio_source"
            ]
            == source
        )
        &
        (
            embedding_df[
                "pseudo_speaker_id"
            ]
            == speaker_id
        )
    )


    indices = (
        embedding_df.loc[
            mask,
            "df_index"
        ]
        .astype(int)
        .tolist()
    )


    if not indices:

        return 0


    df.loc[
        indices,
        "split"
    ] = split_name


    return len(indices)


# ============================================================
# 19. RESET NON-VITOSA
# ============================================================

print("\n" + "=" * 80)
print("RESET NON-VITOSA SOURCES")
print("=" * 80)


non_vitosa_mask = (
    df["audio_source"]
    != BASE_SOURCE
)


df.loc[
    non_vitosa_mask,
    "split"
] = "external_non_toxic"


print(
    "All non-ViToSA samples reset to "
    "'external_non_toxic'."
)


# ============================================================
# 20. SOURCE-BALANCED + TOXICITY-BALANCED SPEAKER ASSIGNMENT
# ============================================================
# Strategy:
#
# 1. ViToSA remains completely unchanged.
#
# 2. ALL FULL SOURCES are assigned jointly, not source-by-source.
#    This is important because one pseudo speaker may occur in
#    multiple FULL SOURCES.
#
# 3. A pseudo speaker is an atomic group:
#       one speaker -> exactly one split.
#
# 4. Assignment minimizes a combined objective:
#       - source balance
#       - toxicity balance
#       - total train/validation balance
#
#    The target train ratio is derived from the original ViToSA
#    train/validation ratio. This avoids imposing an arbitrary 50/50
#    split when ViToSA already has a different protocol.
#
# 5. FULL SOURCE rows are all used whenever a valid speaker embedding
#    exists. No FULL SOURCE row is intentionally discarded because
#    of toxicity or source composition.
#
# 6. FOSD remains a compensation source. It is considered only after
#    FULL SOURCE assignment and only for speakers with non-toxic > toxic.
#
# ============================================================

speaker_assignment = {}


# ------------------------------------------------------------
# Target train/validation ratio from original ViToSA
# ------------------------------------------------------------

vitosa_train_count = int(
    (
        (df["audio_source"] == BASE_SOURCE)
        & (df["split"] == "train")
    ).sum()
)

vitosa_val_count = int(
    (
        (df["audio_source"] == BASE_SOURCE)
        & (df["split"] == "validation")
    ).sum()
)

vitosa_train_val_total = (
    vitosa_train_count + vitosa_val_count
)

if vitosa_train_val_total > 0:
    TARGET_TRAIN_FRACTION = (
        vitosa_train_count
        / vitosa_train_val_total
    )
else:
    TARGET_TRAIN_FRACTION = 0.8

TARGET_VAL_FRACTION = 1.0 - TARGET_TRAIN_FRACTION

# Toxicity target.
# 0.50 means the algorithm prefers approximately balanced toxic /
# non-toxic data while respecting speaker atomicity and source usage.
TARGET_TOXIC_RATIO = 0.50

# Relative weights of the assignment objective.
SOURCE_BALANCE_WEIGHT = 3.0
TOXICITY_BALANCE_WEIGHT = 4.0
TOTAL_BALANCE_WEIGHT = 1.0


print("\n" + "=" * 80)
print("SOURCE-BALANCED + TOXICITY-BALANCED SPEAKER SPLIT")
print("=" * 80)
print(
    f"Target train fraction from ViToSA: "
    f"{TARGET_TRAIN_FRACTION:.4f}"
)
print(
    f"Target validation fraction: "
    f"{TARGET_VAL_FRACTION:.4f}"
)
print(
    f"Target toxic ratio: "
    f"{TARGET_TOXIC_RATIO:.4f}"
)


# ------------------------------------------------------------
# Helper: assign all rows of a speaker across FULL SOURCES
# ------------------------------------------------------------

def assign_full_speaker(
    speaker_id,
    split_name,
):
    """Assign every FULL SOURCE row belonging to one speaker."""

    mask = (
        embedding_df["pseudo_speaker_id"] == speaker_id
    ) & (
        embedding_df["audio_source"].isin(FULL_SOURCES)
    )

    indices = (
        embedding_df.loc[
            mask,
            "df_index",
        ]
        .astype(int)
        .tolist()
    )

    if not indices:
        return 0

    df.loc[
        indices,
        "split",
    ] = split_name

    return len(indices)


# ------------------------------------------------------------
# Build GLOBAL speaker groups for all FULL SOURCES
# ------------------------------------------------------------

full_rows = embedding_df[
    embedding_df["audio_source"].isin(FULL_SOURCES)
].copy()


if full_rows.empty:

    print(
        "[WARNING] No processed FULL SOURCE rows."
    )

else:

    full_speaker_stats = (
        full_rows
        .groupby("pseudo_speaker_id")
        .agg(
            total=("df_index", "count"),
            toxic=("toxicity", "sum"),
            non_toxic=(
                "toxicity",
                lambda x: int((x == 0).sum()),
            ),
            source_count=(
                "audio_source",
                "nunique",
            ),
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # Per-source totals
    # --------------------------------------------------------

    full_source_totals = (
        full_rows
        .groupby("audio_source")
        .size()
        .to_dict()
    )

    full_source_train_targets = {
        source: int(round(
            total * TARGET_TRAIN_FRACTION
        ))
        for source, total
        in full_source_totals.items()
    }

    full_source_val_targets = {
        source: (
            full_source_totals[source]
            - full_source_train_targets[source]
        )
        for source in full_source_totals
    }

    # --------------------------------------------------------
    # Global target counts
    # --------------------------------------------------------

    full_total = int(
        full_rows.shape[0]
    )

    target_full_train_total = int(round(
        full_total * TARGET_TRAIN_FRACTION
    ))

    target_full_val_total = (
        full_total
        - target_full_train_total
    )

    # --------------------------------------------------------
    # Speaker groups can contain multiple sources.
    # Larger / multi-source groups are assigned first so that
    # source diversity is decided early and not left to the tail.
    # --------------------------------------------------------

    full_speaker_stats = (
        full_speaker_stats
        .sort_values(
            by=[
                "source_count",
                "total",
                "non_toxic",
                "toxic",
            ],
            ascending=[
                False,
                False,
                False,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    train_source_counts = {
        source: 0
        for source in full_source_totals
    }
    val_source_counts = {
        source: 0
        for source in full_source_totals
    }

    train_toxic = 0
    train_non_toxic = 0
    val_toxic = 0
    val_non_toxic = 0
    train_total = 0
    val_total = 0


    def speaker_source_counts(speaker_id):
        rows = full_rows[
            full_rows["pseudo_speaker_id"]
            == speaker_id
        ]

        return (
            rows["audio_source"]
            .value_counts()
            .to_dict()
        )


    def split_objective(
        split_name,
        source_counts_after,
        total_after,
        toxic_after,
        non_toxic_after,
    ):
        """Lower is better."""

        if split_name == "train":
            split_fraction = TARGET_TRAIN_FRACTION
            target_total = target_full_train_total
        else:
            split_fraction = TARGET_VAL_FRACTION
            target_total = target_full_val_total

        # ----------------------------------------------------
        # A. Source balance
        # Compare each source's current fraction with its target
        # fraction inside the selected split.
        # ----------------------------------------------------

        source_error = 0.0

        for source, total in full_source_totals.items():
            if total <= 0:
                continue

            expected = (
                total
                * split_fraction
            )

            actual = source_counts_after.get(
                source,
                0,
            )

            source_error += (
                abs(actual - expected)
                / max(total, 1)
            )

        # ----------------------------------------------------
        # B. Toxicity balance
        # ----------------------------------------------------

        current_total = (
            toxic_after
            + non_toxic_after
        )

        if current_total > 0:
            current_toxic_ratio = (
                toxic_after
                / current_total
            )
        else:
            current_toxic_ratio = 0.5

        toxicity_error = abs(
            current_toxic_ratio
            - TARGET_TOXIC_RATIO
        )

        # ----------------------------------------------------
        # C. Total train/validation balance
        # ----------------------------------------------------

        total_error = abs(
            total_after - target_total
        ) / max(
            full_total,
            1,
        )

        return (
            SOURCE_BALANCE_WEIGHT
            * source_error
            + TOXICITY_BALANCE_WEIGHT
            * toxicity_error
            + TOTAL_BALANCE_WEIGHT
            * total_error
        )


    # ------------------------------------------------------------
    # Greedy multi-objective assignment
    # ------------------------------------------------------------

    for _, speaker_row in full_speaker_stats.iterrows():

        speaker_id = speaker_row[
            "pseudo_speaker_id"
        ]

        speaker_total = int(
            speaker_row["total"]
        )
        speaker_toxic = int(
            speaker_row["toxic"]
        )
        speaker_non_toxic = int(
            speaker_row["non_toxic"]
        )

        source_counts = speaker_source_counts(
            speaker_id
        )

        # ----------------------------------------------
        # Evaluate TRAIN
        # ----------------------------------------------

        train_source_after = {
            source: train_source_counts[source]
            for source in train_source_counts
        }

        for source, count in source_counts.items():
            train_source_after[source] += count

        train_score = split_objective(
            "train",
            train_source_after,
            train_total + speaker_total,
            train_toxic + speaker_toxic,
            train_non_toxic + speaker_non_toxic,
        )

        # ----------------------------------------------
        # Evaluate VALIDATION
        # ----------------------------------------------

        val_source_after = {
            source: val_source_counts[source]
            for source in val_source_counts
        }

        for source, count in source_counts.items():
            val_source_after[source] += count

        val_score = split_objective(
            "validation",
            val_source_after,
            val_total + speaker_total,
            val_toxic + speaker_toxic,
            val_non_toxic + speaker_non_toxic,
        )

        # ----------------------------------------------
        # Tie breaker: lower normalized total error
        # ----------------------------------------------

        if abs(train_score - val_score) < 1e-12:

            train_total_error = abs(
                (train_total + speaker_total)
                - target_full_train_total
            )

            val_total_error = abs(
                (val_total + speaker_total)
                - target_full_val_total
            )

            selected_split = (
                "train"
                if train_total_error
                <= val_total_error
                else "validation"
            )

        else:

            selected_split = (
                "train"
                if train_score < val_score
                else "validation"
            )

        # ----------------------------------------------
        # Commit assignment
        # ----------------------------------------------

        speaker_assignment[
            speaker_id
        ] = selected_split

        assigned = assign_full_speaker(
            speaker_id,
            selected_split,
        )

        if selected_split == "train":

            for source, count in source_counts.items():
                train_source_counts[source] += count

            train_total += speaker_total
            train_toxic += speaker_toxic
            train_non_toxic += speaker_non_toxic

        else:

            for source, count in source_counts.items():
                val_source_counts[source] += count

            val_total += speaker_total
            val_toxic += speaker_toxic
            val_non_toxic += speaker_non_toxic


    print("\nFULL SOURCE ASSIGNMENT")
    print("-" * 80)
    print(
        f"Speakers assigned : "
        f"{len(full_speaker_stats):,}"
    )
    print(
        f"Train speakers    : "
        f"{sum(1 for s in full_speaker_stats['pseudo_speaker_id'] if speaker_assignment.get(s) == 'train'):,}"
    )
    print(
        f"Validation speakers: "
        f"{sum(1 for s in full_speaker_stats['pseudo_speaker_id'] if speaker_assignment.get(s) == 'validation'):,}"
    )

    print("\nSOURCE BALANCE")
    print("-" * 80)

    for source in sorted(full_source_totals):
        total = full_source_totals[source]
        train_count = train_source_counts[source]
        val_count = val_source_counts[source]

        print(
            f"{source:25s} | "
            f"total={total:7,} | "
            f"train={train_count:7,} "
            f"({train_count / max(total, 1):.3f}) | "
            f"validation={val_count:7,} "
            f"({val_count / max(total, 1):.3f})"
        )

    print_balance(
        "BALANCE AFTER SOURCE + TOXICITY ASSIGNMENT"
    )


# ============================================================
# 20B. MOVE RETRIEVAL SURPLUS FROM VALIDATION -> TRAIN
# ============================================================
# Keep the original source-balanced + toxicity-balanced split logic.
# This is only a small second-pass adjustment:
#
#   1. Look ONLY at Retrieval speakers currently in validation.
#   2. Move non-toxic-heavy Retrieval speaker groups to TRAIN first.
#   3. Stop when TRAIN reaches the requested non-toxic target.
#   4. Do NOT rebalance Common Voice here.
#   5. Do NOT touch ViToSA.
#   6. If Retrieval validation cannot provide enough non-toxic data,
#      the existing FOSD compensation step below is used as fallback.
#
# Speaker remains atomic: a whole Retrieval speaker group moves.
# This keeps the original split methodology essentially unchanged.
# ============================================================

TRAIN_NON_TOXIC_TARGET_RATIO = 1.20

print("\n" + "=" * 80)
print("RETRIEVAL VALIDATION SURPLUS -> TRAIN")
print("=" * 80)
print(
    f"TRAIN non-toxic target ratio: "
    f"{TRAIN_NON_TOXIC_TARGET_RATIO:.3f}"
)


def get_counts_from_df(split_name):
    subset = df[df["split"] == split_name]
    toxic = int((subset["toxicity"] == 1).sum())
    non_toxic = int((subset["toxicity"] == 0).sum())
    return toxic, non_toxic, toxic + non_toxic


if not full_rows.empty and "retrieval" in FULL_SOURCES:

    train_toxic, train_non_toxic, _ = get_counts_from_df("train")
    train_target_non_toxic = int(
        round(train_toxic * TRAIN_NON_TOXIC_TARGET_RATIO)
    )

    # Only Retrieval speakers that are currently in VALIDATION.
    retrieval_validation_rows = full_rows[
        (full_rows["audio_source"] == "retrieval")
        & (full_rows["pseudo_speaker_id"].isin(
            [
                speaker_id
                for speaker_id, assigned_split
                in speaker_assignment.items()
                if assigned_split == "validation"
            ]
        ))
    ].copy()

    retrieval_val_stats = (
        retrieval_validation_rows
        .groupby("pseudo_speaker_id")
        .agg(
            total=("df_index", "count"),
            toxic=("toxicity", "sum"),
            non_toxic=(
                "toxicity",
                lambda x: int((x == 0).sum()),
            ),
        )
        .reset_index()
    )

    retrieval_val_stats["delta"] = (
        retrieval_val_stats["non_toxic"]
        - retrieval_val_stats["toxic"]
    )

    # Prefer groups that are most useful for increasing TRAIN non-toxic.
    retrieval_val_stats = (
        retrieval_val_stats
        .sort_values(
            by=["delta", "non_toxic", "total"],
            ascending=[False, False, True],
        )
        .reset_index(drop=True)
    )

    moved_speakers = 0
    moved_non_toxic = 0
    moved_total = 0

    for _, row in retrieval_val_stats.iterrows():

        train_toxic, train_non_toxic, _ = get_counts_from_df("train")
        train_target_non_toxic = int(
            round(train_toxic * TRAIN_NON_TOXIC_TARGET_RATIO)
        )
        needed = train_target_non_toxic - train_non_toxic

        if needed <= 0:
            break

        speaker_id = row["pseudo_speaker_id"]
        speaker_non_toxic = int(row["non_toxic"])
        speaker_toxic = int(row["toxic"])

        # Only move non-toxic-beneficial Retrieval groups.
        if speaker_non_toxic <= speaker_toxic:
            continue

        # IMPORTANT: move only the current Retrieval speaker rows.
        # This keeps the previous split logic and source independence intact.
        mask = (
            (embedding_df["audio_source"] == "retrieval")
            & (embedding_df["pseudo_speaker_id"] == speaker_id)
        )
        indices = (
            embedding_df.loc[mask, "df_index"]
            .astype(int)
            .tolist()
        )

        if not indices:
            continue

        df.loc[indices, "split"] = "train"
        speaker_assignment[speaker_id] = "train"

        moved_speakers += 1
        moved_non_toxic += speaker_non_toxic
        moved_total += len(indices)

        print(
            f"[Retrieval validation -> train] {speaker_id} | "
            f"total={len(indices):,} | "
            f"toxic={speaker_toxic:,} | "
            f"non-toxic={speaker_non_toxic:,}"
        )

    final_train_toxic, final_train_non_toxic, final_train_total = (
        get_counts_from_df("train")
    )
    final_val_toxic, final_val_non_toxic, final_val_total = (
        get_counts_from_df("validation")
    )

    print("\nRETRIEVAL REDISTRIBUTION RESULT")
    print("-" * 80)
    print(f"Retrieval speakers moved : {moved_speakers:,}")
    print(f"Samples moved            : {moved_total:,}")
    print(f"Non-toxic moved          : {moved_non_toxic:,}")
    print(
        f"TRAIN       total={final_train_total:,} | "
        f"toxic={final_train_toxic:,} | "
        f"non-toxic={final_train_non_toxic:,} | "
        f"ratio={final_train_non_toxic / max(final_train_toxic, 1):.3f}"
    )
    print(
        f"VALIDATION  total={final_val_total:,} | "
        f"toxic={final_val_toxic:,} | "
        f"non-toxic={final_val_non_toxic:,} | "
        f"ratio={final_val_non_toxic / max(final_val_toxic, 1):.3f}"
    )

else:
    print("[INFO] Retrieval not available; skipping Retrieval redistribution.")


# ============================================================
# 21. FOSD COMPENSATION
# ============================================================

print("\n" + "=" * 80)
print("FOSD - TOXICITY COMPENSATION ONLY")
print("=" * 80)

fosd_rows = embedding_df[
    embedding_df["audio_source"] == FOSD_SOURCE
].copy()


if fosd_rows.empty:

    print(
        "[WARNING] No processed FOSD samples."
    )

else:

    print(
        f"FOSD processed samples: "
        f"{len(fosd_rows):,}"
    )

    fosd_speakers = (
        fosd_rows[
            "pseudo_speaker_id"
        ]
        .dropna()
        .unique()
        .tolist()
    )

    # A FOSD speaker already used by a FULL SOURCE is locked to
    # that split and must not be reused independently.
    fosd_candidate_speakers = [
        speaker_id
        for speaker_id in fosd_speakers
        if speaker_id
        not in speaker_assignment
    ]

    fosd_candidates = (
        fosd_rows[
            fosd_rows[
                "pseudo_speaker_id"
            ].isin(
                fosd_candidate_speakers
            )
        ]
        .groupby(
            "pseudo_speaker_id"
        )
        .agg(
            total=(
                "df_index",
                "count"
            ),
            toxic=(
                "toxicity",
                "sum"
            ),
            non_toxic=(
                "toxicity",
                lambda x: int(
                    (x == 0).sum()
                )
            ),
        )
        .reset_index()
    )

    fosd_candidates["delta"] = (
        fosd_candidates["non_toxic"]
        - fosd_candidates["toxic"]
    )

    # Only speakers that improve the non-toxic side are eligible.
    fosd_candidates = (
        fosd_candidates[
            fosd_candidates["delta"] > 0
        ]
        .sort_values(
            by=[
                "delta",
                "non_toxic",
                "total",
            ],
            ascending=[
                False,
                False,
                True,
            ]
        )
        .reset_index(drop=True)
    )

    print(
        f"FOSD eligible speakers: "
        f"{len(fosd_candidates):,}"
    )


    def apply_fosd_compensation(
        split_name,
    ):
        """Add FOSD speakers until toxic/non-toxic is balanced."""

        toxic, non_toxic, _ = get_split_counts(
            split_name
        )

        deficit = max(
            0,
            toxic - non_toxic,
        )

        print(
            f"\n{split_name.upper()} "
            f"non-toxic deficit before FOSD: "
            f"{deficit:,}"
        )

        if deficit <= 0:
            return

        gained = 0

        for _, row in fosd_candidates.iterrows():

            if gained >= deficit:
                break

            speaker_id = row[
                "pseudo_speaker_id"
            ]

            if speaker_id in speaker_assignment:
                continue

            speaker_toxic = int(
                row["toxic"]
            )
            speaker_non_toxic = int(
                row["non_toxic"]
            )
            delta = (
                speaker_non_toxic
                - speaker_toxic
            )

            speaker_assignment[
                speaker_id
            ] = split_name

            assigned = assign_source_speaker(
                FOSD_SOURCE,
                speaker_id,
                split_name,
            )

            gained += delta

            print(
                f"[FOSD -> {split_name}] "
                f"{speaker_id} | "
                f"samples={assigned:,} | "
                f"toxic={speaker_toxic:,} | "
                f"non-toxic={speaker_non_toxic:,} | "
                f"delta={delta:+,}"
            )


    apply_fosd_compensation("train")
    apply_fosd_compensation("validation")


# ============================================================
# 22. ALL REMAINING FOSD -> EXTERNAL
# ============================================================

if not fosd_rows.empty:

    fosd_indices = set(
        fosd_rows[
            "df_index"
        ]
        .astype(int)
        .tolist()
    )

    fosd_train_val_indices = set(
        df.loc[
            (
                df["audio_source"]
                == FOSD_SOURCE
            )
            & (
                df["split"].isin(
                    [
                        "train",
                        "validation",
                    ]
                )
            )
        ]
        .index
        .astype(int)
        .tolist()
    )

    remaining_fosd_indices = (
        fosd_indices
        - fosd_train_val_indices
    )

    if remaining_fosd_indices:

        df.loc[
            list(remaining_fosd_indices),
            "split"
        ] = "external_non_toxic"

    print(
        "\nRemaining FOSD reserve: "
        f"{len(remaining_fosd_indices):,}"
    )


# ============================================================
# 23. SOURCE-BALANCE SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("FINAL SOURCE BALANCE - TRAIN / VALIDATION")
print("=" * 80)

for source in FULL_SOURCES:

    source_total = int(
        (
            df["audio_source"]
            == source
        ).sum()
    )

    source_train = int(
        (
            (df["audio_source"] == source)
            & (df["split"] == "train")
        ).sum()
    )

    source_val = int(
        (
            (df["audio_source"] == source)
            & (df["split"] == "validation")
        ).sum()
    )

    source_external = int(
        (
            (df["audio_source"] == source)
            & (
                df["split"]
                == "external_non_toxic"
            )
        ).sum()
    )

    print(
        f"{source:25s} | "
        f"total={source_total:7,} | "
        f"train={source_train:7,} "
        f"({source_train / max(source_total, 1):.3f}) | "
        f"validation={source_val:7,} "
        f"({source_val / max(source_total, 1):.3f}) | "
        f"external={source_external:7,}"
    )


# ============================================================
# 24. FINAL TOXICITY BALANCE SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("FINAL TOXICITY BALANCE")
print("=" * 80)

for split_name in [
    "train",
    "validation",
]:

    toxic, non_toxic, total = get_split_counts(
        split_name
    )

    print(
        f"{split_name:12s} | "
        f"total={total:,} | "
        f"toxic={toxic:,} | "
        f"non-toxic={non_toxic:,} | "
        f"non-toxic/toxic="
        f"{non_toxic / max(toxic, 1):.3f}"
    )

# 25. VERIFY ViToSA WAS NOT CHANGED
# ============================================================

print("\n" + "=" * 80)
print("VERIFY VITOSA SPLIT")
print("=" * 80)


original_vitosa = (
    pd.read_csv(INPUT_CSV)
    .reset_index(drop=True)
)


original_vitosa_mask = (
    original_vitosa[
        "audio_source"
    ]
    == BASE_SOURCE
)


current_vitosa_mask = (
    df[
        "audio_source"
    ]
    == BASE_SOURCE
)


original_vitosa_splits = (
    original_vitosa.loc[
        original_vitosa_mask,
        "split"
    ]
    .reset_index(drop=True)
)


current_vitosa_splits = (
    df.loc[
        current_vitosa_mask,
        "split"
    ]
    .reset_index(drop=True)
)


if original_vitosa_splits.equals(
    current_vitosa_splits
):

    print(
        "OK: ViToSA split unchanged."
    )

else:

    print(
        "WARNING: ViToSA split changed!"
    )


# ============================================================
# 26. FINAL BALANCE
# ============================================================

print_balance(
    "FINAL TRAIN / VALIDATION BALANCE"
)


# ============================================================
# 27. FINAL SOURCE × SPLIT
# ============================================================

print("\n" + "=" * 80)
print("FINAL SOURCE × SPLIT")
print("=" * 80)


source_split = pd.crosstab(
    df["audio_source"],
    df["split"],
    margins=True,
    margins_name="Tổng_Cộng"
)


print(
    source_split
)


# ============================================================
# 28. FINAL SOURCE × TOXICITY
# ============================================================

print("\n" + "=" * 80)
print("FINAL SOURCE × TOXICITY")
print("=" * 80)


source_toxicity = pd.crosstab(
    df["audio_source"],
    df["toxicity"],
    margins=True,
    margins_name="Tổng_Cộng"
)


print(
    source_toxicity
)


# ============================================================
# 29. FINAL SPLIT × TOXICITY
# ============================================================

print("\n" + "=" * 80)
print("FINAL SPLIT × TOXICITY")
print("=" * 80)


split_toxicity = pd.crosstab(
    df["split"],
    df["toxicity"],
    margins=True,
    margins_name="Tổng_Cộng"
)


print(
    split_toxicity
)


# ============================================================
# 30. SOURCE × SPLIT × TOXICITY
# ============================================================

print("\n" + "=" * 80)
print("SOURCE × SPLIT × TOXICITY")
print("=" * 80)


full_crosstab = pd.crosstab(
    [
        df["audio_source"],
        df["split"],
    ],
    df["toxicity"],
    margins=True,
    margins_name="Tổng_Cộng"
)


print(
    full_crosstab
)


# ============================================================
# 31. SPEAKER INDEPENDENCE CHECK
# ============================================================

train_speaker_ids = set(
    df.loc[
        df["split"] == "train",
        "pseudo_speaker_id"
    ]
    .dropna()
    .astype(str)
)


val_speaker_ids = set(
    df.loc[
        df["split"] == "validation",
        "pseudo_speaker_id"
    ]
    .dropna()
    .astype(str)
)


speaker_overlap = (
    train_speaker_ids
    &
    val_speaker_ids
)


print("\n" + "=" * 80)
print("SPEAKER INDEPENDENCE CHECK")
print("=" * 80)


print(
    f"Train speakers      : "
    f"{len(train_speaker_ids):,}"
)


print(
    f"Validation speakers : "
    f"{len(val_speaker_ids):,}"
)


print(
    f"Speaker overlap     : "
    f"{len(speaker_overlap):,}"
)


if speaker_overlap:

    print(
        "❌ WARNING: "
        "Speaker leakage detected!"
    )

    print(
        "\nOverlapping speakers:"
    )

    for speaker_id in sorted(
        speaker_overlap
    )[:50]:

        print(
            f"  {speaker_id}"
        )

else:

    print(
        "✅ No speaker overlap between "
        "train and validation."
    )


# ============================================================
# 32. SPEAKER MULTI-SPLIT CHECK
# ============================================================

print("\n" + "=" * 80)
print("SPEAKER MULTI-SPLIT CHECK")
print("=" * 80)


speaker_split_counts = (
    df[
        df["pseudo_speaker_id"]
        .notna()
    ]
    .groupby(
        "pseudo_speaker_id"
    )["split"]
    .nunique()
)


multi_split_speakers = (
    speaker_split_counts[
        speaker_split_counts > 1
    ]
)


print(
    f"Speakers with >1 split: "
    f"{len(multi_split_speakers):,}"
)


if len(multi_split_speakers) == 0:

    print(
        "✅ Every pseudo speaker belongs "
        "to only one split."
    )

else:

    print(
        "❌ WARNING: Some speakers occur "
        "in multiple splits."
    )


# ============================================================
# 33. FULL SOURCE USAGE CHECK
# ============================================================

print("\n" + "=" * 80)
print("FULL SOURCE USAGE CHECK")
print("=" * 80)


for source in FULL_SOURCES:

    total_count = int(
        (
            df["audio_source"]
            == source
        ).sum()
    )


    train_val_count = int(
        (
            (
                df["audio_source"]
                == source
            )
            &
            (
                df["split"].isin(
                    [
                        "train",
                        "validation",
                    ]
                )
            )
        ).sum()
    )


    external_count = int(
        (
            (
                df["audio_source"]
                == source
            )
            &
            (
                df["split"]
                == "external_non_toxic"
            )
        ).sum()
    )


    print(
        f"{source:25s} | "
        f"total={total_count:7,} | "
        f"train/val={train_val_count:7,} | "
        f"external={external_count:7,}"
    )


    if train_val_count == total_count:

        print(
            " " * 4
            + "✅ FULL SOURCE completely used."
        )

    else:

        print(
            " " * 4
            + "⚠️ FULL SOURCE has unused rows."
        )


# ============================================================
# 34. FOSD USAGE CHECK
# ============================================================

if FOSD_SOURCE in available_sources:

    print("\n" + "=" * 80)
    print("FOSD USAGE CHECK")
    print("=" * 80)


    fosd_total = int(
        (
            df["audio_source"]
            == FOSD_SOURCE
        ).sum()
    )


    fosd_train = int(
        (
            (
                df["audio_source"]
                == FOSD_SOURCE
            )
            &
            (
                df["split"]
                == "train"
            )
        ).sum()
    )


    fosd_val = int(
        (
            (
                df["audio_source"]
                == FOSD_SOURCE
            )
            &
            (
                df["split"]
                == "validation"
            )
        ).sum()
    )


    fosd_external = int(
        (
            (
                df["audio_source"]
                == FOSD_SOURCE
            )
            &
            (
                df["split"]
                == "external_non_toxic"
            )
        ).sum()
    )


    fosd_other = int(
        (
            (
                df["audio_source"]
                == FOSD_SOURCE
            )
            &
            (
                ~df["split"].isin(
                    [
                        "train",
                        "validation",
                        "external_non_toxic",
                    ]
                )
            )
        ).sum()
    )


    print(
        f"FOSD total             : "
        f"{fosd_total:,}"
    )


    print(
        f"FOSD -> train          : "
        f"{fosd_train:,}"
    )


    print(
        f"FOSD -> validation     : "
        f"{fosd_val:,}"
    )


    print(
        f"FOSD -> external       : "
        f"{fosd_external:,}"
    )


    print(
        f"FOSD -> other          : "
        f"{fosd_other:,}"
    )


    print(
        f"Check total            : "
        f"{fosd_train + fosd_val + fosd_external + fosd_other:,}"
    )


    if (
        fosd_train
        + fosd_val
        + fosd_external
        + fosd_other
        == fosd_total
    ):

        print(
            "✅ FOSD accounting is consistent."
        )

    else:

        print(
            "❌ FOSD accounting mismatch!"
        )


# ============================================================
# 35. CHECK TRAIN / VALIDATION BALANCE REQUIREMENT
# ============================================================

print("\n" + "=" * 80)
print("NON-TOXIC BALANCE REQUIREMENT")
print("=" * 80)


for split_name in [
    "train",
    "validation",
]:

    toxic, non_toxic, total = (
        get_split_counts(
            split_name
        )
    )


    print(
        f"{split_name:12s} | "
        f"toxic={toxic:,} | "
        f"non-toxic={non_toxic:,} | "
        f"non-toxic >= toxic: "
        f"{non_toxic >= toxic}"
    )


# ============================================================
# 36. FINAL SPLIT DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("FINAL SPLIT DISTRIBUTION")
print("=" * 80)


print(
    df["split"]
    .value_counts(
        dropna=False
    )
)


# ============================================================
# 37. CHECK MISSING SPLIT
# ============================================================

missing_split_count = int(
    df["split"]
    .isna()
    .sum()
)


print(
    f"\nMissing split: "
    f"{missing_split_count:,}"
)


if missing_split_count == 0:

    print(
        "✅ No missing split."
    )

else:

    print(
        "❌ WARNING: "
        "Some rows have missing split."
    )


# ============================================================
# 38. CHECK pseudo_speaker_id
# ============================================================

missing_speaker_count = int(
    df[
        df["audio_source"]
        != BASE_SOURCE
    ][
        "pseudo_speaker_id"
    ]
    .isna()
    .sum()
)


print(
    f"Non-ViToSA rows without "
    f"pseudo speaker: "
    f"{missing_speaker_count:,}"
)


# ============================================================
# 39. SAVE
# ============================================================

print("\n" + "=" * 80)
print("SAVING")
print("=" * 80)


df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"Output:\n"
    f"{OUTPUT_CSV}"
)


print(
    f"Total rows: "
    f"{len(df):,}"
)


# ============================================================
# 40. FINAL SUCCESS MESSAGE
# ============================================================

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)


if len(speaker_overlap) == 0:

    print(
        "✅ Speaker independence: PASS"
    )

else:

    print(
        "❌ Speaker independence: FAIL"
    )


if missing_split_count == 0:

    print(
        "✅ Split completeness: PASS"
    )

else:

    print(
        "❌ Split completeness: FAIL"
    )


print(
    "\nFinal dataset saved successfully."
)

a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA SpeechRun\envLab\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE
Device: cuda
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU

LOADING DATASET
Total samples: 32,303

AUDIO SOURCE
audio_source
vitosa          12802
fosd            12600
retrieval        4542
common_voice     2359
Name: count, dtype: int64

ORIGINAL SPLIT
split
external_non_toxic    14143
train                 13183
validation             2161
test                   2000
NaN                     816
Name: count, dtype: int64

SOURCE × TOXICITY
toxicity          0      1  Tổng_Cộng
audio_source                         
common_voice   2073    286       2359
fosd          12070    530      12600
retrieval      4542      0       4542
vitosa         1000  11802      12802
Tổng_Cộng     19685  12618      32303

SOURCE CONFIGURATION
Base source: vitosa

FULL SOURCES:
  01. common_voice
  02. retrieval

COMPENSATION SOURCE:
  fosd

TARGET DATA FOR SPEAKER CLUSTERING
Target samples: 19,501
audio_source
fosd            12600
retrieval        4542
common_voice     2359
Name: c

a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA SpeechRun\envLab\Lib\site-packages\speechbrain\utils\checkpoints.py:202: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  stat

ECAPA-TDNN loaded successfully.

EXTRACTING SPEAKER EMBEDDINGS


ECAPA: 100%|██████████| 19501/19501 [06:22<00:00, 51.05it/s]



EMBEDDING EXTRACTION COMPLETED
Valid embeddings : 19,457
Skipped audio    : 44

Skipped report saved:
A:\A _ Working\Researching\B - AIoT Lab VN\VITOSA SpeechRun\vitosa_datasets\speaker_split_skipped_audio5.1.2.csv

Embedding shape:
(19457, 192)

CLUSTERING SPEAKERS
Detected pseudo speakers: 5,484

SPEAKER SUMMARY
   pseudo_speaker_id  samples       sources  toxic  non_toxic  delta
0      SPEAKER_00000        8          fosd      0          8      8
1      SPEAKER_00001       12     retrieval      0         12     12
2      SPEAKER_00002        2     retrieval      0          2      2
3      SPEAKER_00003       11          fosd      0         11     11
4      SPEAKER_00004       16          fosd      1         15     14
5      SPEAKER_00005        4     retrieval      0          4      4
6      SPEAKER_00006        8     retrieval      0          8      8
7      SPEAKER_00007       19          fosd      1         18     17
8      SPEAKER_00008        2          fosd      0          2 

# show split 5.1.2

In [14]:
import pandas as pd

datasets = pd.read_csv(
    r"A:\A _ Working\Researching\B - AIoT Lab VN\VITOSA SpeechRun\vitosa_datasets\final_vietnamese_toxic_utterance_dataset_v5.1.2_speaker_split.csv"
)

# ============================================================
# THỐNG KÊ AUDIO_SOURCE × SPLIT × TOXICITY
# ============================================================

print("\n--- THỐNG KÊ AUDIO_SOURCE × SPLIT × TOXICITY ---")

cross_stats = pd.crosstab(
    index=datasets["audio_source"],
    columns=[
        datasets["split"],
        datasets["toxicity"]
    ],
    margins=True,
    margins_name="Tổng_Cộng"
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", 2000
):
    print(cross_stats)


--- THỐNG KÊ AUDIO_SOURCE × SPLIT × TOXICITY ---
split        external_non_toxic       test       train       validation       Tổng_Cộng
toxicity                      0    1     0     1     0     1          0     1          
audio_source                                                                           
common_voice                  0    0     0     0   310    96       1763   190      2359
fosd                       7319  291     0     0  4116   209        635    30     12600
retrieval                     0    0     0     0  4542     0          0     0      4542
vitosa                        0    0  1000  1000     0  8641          0  2161     12802
Tổng_Cộng                  7319  291  1000  1000  8968  8946       2398  2381     32303
